# Generates dataset of high quality probes


In [1]:
# importing modules
from tools import ProbesMainSet, ChemblDB, PnDDB
import os
import pandas as pd

### Databases
Both databases Probes&Drugs and ChEMBL are loaded from the SQLite dumps and processed in tools.py

They can be downloaded from:

* **ChEMBL SQLite dump** https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/chembl_33_sqlite.tar.gz
* **Probes&Drugs SQLite dump** https://www.probes-drugs.org/media/download/dump/pd_01_2023_sqlite.zip

Then added to the mainfolder path.

In [ ]:
#define path to main folder
mainfolder = 'files'

#Define paths to the sqlite versions of databases
chemblpath = 'chembl_35/chembl_35_sqlite/chembl_35.db' #replace your own path to chembl.db sqlite dump
pndpath = os.path.join(mainfolder,'pd_02_2024.sqlite')
print(chemblpath, pndpath)

#Initialize database
chembl = ChemblDB(chemblpath) #ChEMBL SQLite dump https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/chembl_33_sqlite.tar.gz
pnd = PnDDB(pndpath) #Probes&Drugs SQLite dump https://www.probes-drugs.org/media/download/dump/pd_01_2023_sqlite.zip

#Define paths to filtering files
CPportalpath = os.path.join(mainfolder, 'ChemicalProbesPortal-07_01_2025.csv') #filtered file from chemicalprobes.org

files/chembl_33_sqlite/chembl_33.db files/pd_02_2024.sqlite


In [5]:
#getting the mainset instance with general data (full datasets) from Probe&Drugs database
mainset = ProbesMainSet(pnd,chembl) #instance of pnd DB class, instance of chembl DB class, path of putput file
mainset.clean_data() #merge synonyms and deletes duplicates
mainset.write_mainset(os.path.join(mainfolder,'probesMainset.csv')) #writes the csv output file
alldata = mainset.mainset #assigns to a variable

In [9]:
#show data format
display(alldata.head(2))

,SETID,UNIPROT,PROBE,SET,INCHI,GENE,CONTR,CHEMBLID,PREF_NAME,SYNOMS,SYNOMS_TARGET
0,28,Q7JJ13,(+)-JQ1,SGC Probes,DNVXATUJJDPFDM-KRWDZBQOSA-N,BRD2,(-)-JQ1,CHEMBL1957266,None,"(S)-JQ1, (+)-JQ1, (+)-JQ-1, JQ1, JQ-1, 1268524...","Female sterile homeotic-related protein 1, FSR..."
1,408,Q7JJ13,(+)-JQ1,Chemical Probes.org,DNVXATUJJDPFDM-KRWDZBQOSA-N,BRD2,(-)-JQ1,CHEMBL1957266,None,"(S)-JQ1, (+)-JQ1, (+)-JQ-1, JQ1, JQ-1, 1268524...","Female sterile homeotic-related protein 1, FSR..."


In [10]:
#some numbers for probes inchikeys
print(alldata[alldata.SETID == 408].INCHI.nunique()) #chemicalprobes.org all unique inchikeys
print(alldata[alldata.SETID == 28].INCHI.nunique()) #SGC all unique inchikeys
print(alldata[alldata.SETID == 213].INCHI.nunique()) #OpenScience all unique inchikeys
print(alldata.INCHI.nunique()) #all unique inchikeys in dataset before filtering (**sources have inchikeys in common)

619
88
100
647


In [11]:
#filtering data by good quality probes in ChemicalProbes.org
CPportal = pd.read_csv(CPportalpath, sep=',') #Reads files from chemicalprobes.org with filtered probes
CPinchis = CPportal['InChI key'].unique().tolist() #gets unique inchikeys from good quality set in chemicalprobes.org
notCPinchis = alldata[alldata.SET != 'Chemical Probes.org'].INCHI.unique().tolist() #gets all inchikeys from SGC and open sciencd
allinchis = list(set(notCPinchis + CPinchis)) #merge inchis from SGC, openScience, and good set in chemicalprobes.org.
subset=alldata[alldata.INCHI.isin(allinchis)]

print(subset.INCHI.nunique()) #all inchikeys after filtering

398


In [12]:
#Adding subset with filtered data to the mainset class instance
mainset.subset = subset
mainset.write_subset(os.path.join(mainfolder,'probesSubset.csv'))
